In [1]:
INDEX = ["AAPL", "GLD", "INTC", "SPY"]

from fts_tool.config import PRICES_DATA_SOURCE
from fts_tool.ingestion import YahooFinanceIngestor
import pandas as pd
import numpy as np

prices_df = {}

for ticker in INDEX:
    yf_ingestor = YahooFinanceIngestor(ticker=ticker, start_date="2018-01-01", end_date="2023-01-01", interval="1d")
    data = yf_ingestor.fetch_data()
    processed_data = yf_ingestor.process_data(data)

    yf_ingestor.store_data(processed_data)

    prices_df[ticker] = processed_data

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [2]:
AAPL_prices = prices_df["AAPL"].Close.sort_index()
returns = AAPL_prices.pct_change().dropna()
log_returns = np.log(AAPL_prices / AAPL_prices.shift(1)).dropna()

# 2. Simple Linear Times Series

Here, we will study stationarity, white noise, simple linear times series like MA, RA...


- Stationarity
- White Noise
- MA
- AR (coef, ACF, average lenght of stochastic cycles, order determination)
- ARMA
- Unit-Root NonStationnarity
- Season Models
- Regression Models with Time Series Errors
- Consistent covariance matrix estimation

## Stationarity

In finance literature, it is common to assume that an asset return series is weakly srationnary. This assumption can be checked empiricazlly provided that a sufficient number of historical returns are available. For example, one can divide the data into subsamples and check the consistency of the results obtained across the subsamples.

In [4]:
from fts_tool.stationarity import StationarityTester

stationarity_tester = StationarityTester(log_returns)

# Test for stationarity using the Augmented Dickey-Fuller test
adf_result = stationarity_tester.adf_test()

print("ADF Statistic:", adf_result["statistic"])
print("p-value:", adf_result["p_value"])
#print("Critical Values:", adf_result["Critical Values"])

ADF Statistic: -14.028026474177844
p-value: 3.4749507642810807e-26


In [5]:
# Test for stationarity using the KPSS test
kpss_result = stationarity_tester.kpss_test()

print("KPSS Statistic:", kpss_result["statistic"])
print("p-value:", kpss_result["p_value"])
#print("Critical Values:", kpss_result["critical_values"])

KPSS Statistic: 0.14670630839643076
p-value: 0.1


c:\Users\chris\Desktop\Price Modelling\financial_times_series\fts_tool\stationarity.py:67: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  result = kpss(


In [6]:
# Test for stationarity using Zivot-Andrews test
za_result = stationarity_tester.zivot_andrews_test()

print("Zivot-Andrews Statistic:", za_result["statistic"])
print("p-value:", za_result["p_value"])

Zivot-Andrews Statistic: -14.332424263943134
p-value: 1e-05


### Small Comment

For Apple over 2019–2023, the tests give the expected result for both simple and log returns: ADF and Zivot–Andrews reject a unit root, while KPSS does not reject stationarity around a constant. This applies to the **return series**, not necessarily to the stock price.

These full-sample results are a useful starting point, but they may hide changes over time. A natural next step is to repeat the tests on subperiods or rolling windows to see whether the stationarity diagnostics remain stable across different market conditions.

### 📌 Stationarity diagnostics: ADF, KPSS and Zivot–Andrews

A financial time series can look non-stationary because shocks accumulate over time, because it follows a deterministic trend, or because its level or trend changes at a particular date. These three tests help distinguish some of those explanations, but none can identify the data-generating process with certainty.

| Test | Null hypothesis \(H_0\) | What rejection suggests |
| --- | --- | --- |
| ADF | The series has a unit root. | Evidence against a unit root under the selected constant/trend specification. |
| KPSS | The series is stationary around a level or deterministic trend. | Evidence against that form of stationarity. |
| Zivot–Andrews | The series has a unit root in a model allowing for one structural break. | Evidence against the unit root, in favour of a specification that allows one break. |

**Important:** Not rejecting \(H_0\) is not proof that \(H_0\) is true. Also, “stationary” must be read relative to the model: stationary around a constant, around a trend, or around a *broken* deterministic component are different statements.

#### 1. ADF — Is there evidence against a unit root?

The Augmented Dickey–Fuller regression is commonly written as

\[
\Delta Y_t
= \alpha + \beta t + \gamma Y_{t-1}
+ \sum_{i=1}^{p}\delta_i\Delta Y_{t-i}
+ \varepsilon_t.
\]

The constant \(\alpha\) and trend \(\beta t\) depend on the chosen specification. The lagged differences account for serial dependence.

- \(H_0:\gamma=0\): the series has a unit root.
- \(H_1:\gamma<0\): evidence against a unit root under the selected specification.
- `regression="c"` includes a constant; `regression="ct"` includes a constant and a linear trend.

**Intuition.** In the simple model \(Y_t=\phi Y_{t-1}+\varepsilon_t\), \(\phi=1\) gives a random walk: shocks accumulate in the level. If \(|\phi|<1\), their effects decay. This AR(1) example is intuition, not the full ADF model.

**Limitation.** If the series changes level or trend and the regression ignores that break, ADF can have difficulty rejecting the unit-root null. A large p-value does not establish that the series is a random walk.

#### 2. KPSS — Is stationarity plausible as the null?

KPSS starts from the opposite hypothesis. One way to describe its model is

\[
Y_t=\alpha+\beta t+r_t+\varepsilon_t,
\qquad
r_t=r_{t-1}+u_t.
\]

Here, \(r_t\) is a stochastic-trend component and \(\varepsilon_t\) is stationary.

- \(H_0:\operatorname{Var}(u_t)=0\): no stochastic trend; the series is stationary around the selected deterministic component.
- \(H_1:\operatorname{Var}(u_t)>0\): evidence of a stochastic trend.
- `regression="c"` tests stationarity around a constant; `regression="ct"` tests stationarity around a deterministic linear trend.

KPSS uses cumulative sums of residuals after removing the selected level or trend. Persistent deviations make its statistic larger. A level or trend break that the model does not include can therefore lead KPSS to reject stationarity around **one unchanged** level or trend.

> KPSS p-values in `statsmodels` come from a reference table. If a statistic falls outside its range, the returned p-value is a boundary value, not a precise estimate; read the accompanying warning.

#### 3. Zivot–Andrews — Does allowing one break change the unit-root diagnosis?

Zivot–Andrews is a unit-root test that searches for **one break date** rather than requiring us to specify that date in advance. Its model can allow a change in the level, the deterministic trend, or both.

- \(H_0\): a unit root under the test's structural-break specification.
- \(H_1\): evidence against the unit root in favour of a specification allowing one break.
- `regression="c"` allows a break in the intercept; `"t"` allows a break in the trend; `"ct"` allows both.
- The reported `break_position` is the estimated position of the break **within the series passed to the test**. It is an estimate, not proof of a known historical event.

Zivot–Andrews is useful when an ordinary ADF result may be affected by an omitted break. It does **not** establish that every large price move is a structural break, and its one-break specification may be inadequate if the sample contains several regime changes.

#### How to read the tests together

| ADF | KPSS | Initial interpretation |
| --- | --- | --- |
| Rejects unit root | Does not reject stationarity | Consistent with stationarity under the chosen specifications. |
| Does not reject unit root | Rejects stationarity | Consistent with a unit root, but not conclusive. |
| Does not reject unit root | Does not reject stationarity | Inconclusive: neither null was rejected. |
| Rejects unit root | Rejects stationarity | Conflicting evidence: inspect specifications, breaks and sample periods. |

If ADF does not reject a unit root, **Zivot–Andrews can add a targeted check**: does the result change when one structural break is allowed? Do not treat its p-value as an automatic tie-breaker; the tests examine different specifications.

#### Practical workflow for financial data

1. Specify the series being tested: prices, log-prices or returns. A result for one does not automatically apply to another.
2. Plot the data and decide whether a constant (`"c"`) or a deterministic trend (`"ct"`) is sensible for ADF and KPSS.
3. Compare ADF and KPSS, recording their p-values, critical values and lag choices.
4. If a break is plausible, inspect the estimated Zivot–Andrews break date and its economic context. Choose `"c"`, `"t"` or `"ct"` according to the type of break you want the model to allow.
5. Check whether the interpretation survives reasonable changes in sample period and specification.

A visually dramatic market crash is a reason to **investigate** a structural break—not evidence, by itself, that the process became non-stationary or that a unit root is present.

### White Noise